In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [2]:
person_ids <- read.csv('data/person_ids.csv', header = TRUE)

In [4]:
exclusion_data = "CB_2489.cb_ExclusionStudentsDetails"

exclusion_table <- tbl(con, exclusion_data) |>
    select(person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,PrimarySENtype) 

In [6]:
exclusion_df <- collect(exclusion_table)

# Extract SEN type to compare with missing

In [15]:
exclusion_df |> distinct(PrimarySENtype) |> pull(PrimarySENtype)

[1] NA     " "    "BESD" "OTH"  "MLD"  "HI"   "SPLD" "SLCN" "PMLD" "ASD" 
[11] "PD"   "SLD"  "SEMH" "NSA"  "VI"   "MSI"

In [38]:
sen <- exclusion_df |>
    select(person_id, PrimarySENtype) |>
    filter(PrimarySENtype %in% c('BESD','OTH','MLD','HI','SPLD','SLCN','PMLD','ASD','PD','SLD','SEMH','NSA','VI','MSI'))


In [39]:
sen |> nrow()

[1] 10927

## Save csv

In [40]:
# save as csv
write.csv(sen, "data/sen_extras.csv", row.names = FALSE)

# Exclusions

In [8]:
exclusion_df |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2002" "2003" "2004" "2005" "2006" "2007" "2008" "2009" "2010" "2011"
[11] "2012" "2013" "2014" "2015" "2016" "2017" "2018" "2019" "2020" "2021"

In [9]:
exclusion_df |> arrange(NumberOfEnrolments) |> distinct(NumberOfEnrolments) |> pull(NumberOfEnrolments)

[1]  1  2  3  4 NA

In [17]:
exclusions_filtered <- exclusion_df |>
    filter(person_id %in% person_ids$person_id) |>
    select(-PrimarySENtype)

In [18]:
exclusions_filtered |> nrow()

[1] 5037

In [19]:
exclusions_filtered |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018" "2019" "2020"

In [20]:
exclusions_filtered_cohort <- exclusions_filtered |>
    left_join(person_ids, by = join_by(person_id))

In [22]:
head(exclusions_filtered_cohort)

person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,NCCIS_ACADYR
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
6FB5E22CD8825E788A477B4CCEEBAC0B4A50D56141F9322659699841DCC1E97C,2007,1,15,41,NA,2017/2018
B353E64ABC7B6CEE2C89B379F6E1FE1B7C11957CD8A50E95BC2F2BFA858CF444,2007,1,11,59,NA,2018/2019
AC5A4C888ECAE0D44D1208E14F61E95B3BC90FB52AA09FD6C246773C77A43E26,2017,1,14,31,0,2017/2018
8C35711901E2F60373EF9AC887E97C95765384613B3F440D8D190C7F2205417C,2017,1,3,40,0,2017/2018
26EC80FD819A62ED03E3A9B47BB5F3F72807675A7B8079A1A392629CBD033260,2017,1,2,32,0,2018/2019
B6526B45A48C10CE38A88A086DB9B3A340C7D2E8DFB72759522CF4575BC19191,2017,1,3,36,0,2018/2019


## Select dates by cohort

In [23]:
cohort_1 <- exclusions_filtered_cohort |> filter(NCCIS_ACADYR == '2017/2018')

In [25]:
cohort_1 |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018" "2019"

In [27]:
cohort_1 <- cohort_1 |>
    filter(Year %in% c('2012','2013','2014','2015','2016','2017'))

In [24]:
cohort_2 <- exclusions_filtered_cohort |> filter(NCCIS_ACADYR == '2018/2019')

In [26]:
cohort_2 |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018" "2019" "2020"

In [28]:
cohort_2 <- cohort_2 |>
    filter(Year %in% c('2013','2014','2015','2016','2017','2018'))

In [29]:
cohort_2 |> arrange(person_id, Year)

person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,NCCIS_ACADYR
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,2017,1,4,21,0,2018/2019
002EF3F17C7981C3D6FD842C559E9AA0626E06B3940B670DDA2B71F7070ADF7D,2017,1,2,12,0,2018/2019
002EF3F17C7981C3D6FD842C559E9AA0626E06B3940B670DDA2B71F7070ADF7D,2018,1,1,2,0,2018/2019
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,2013,1,5,10,0,2018/2019
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,2014,1,13,35,0,2018/2019
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,2015,1,1,6,0,2018/2019
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,2016,1,6,90,0,2018/2019
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,2017,1,1,2,0,2018/2019
00DE1C30EB1A1AA2CE3FB40A6E8F2CD3C88180EE30C0AE6DEB98D3F0D1BD812C,2016,1,1,1,0,2018/2019


## Rejoin cohorts and tally exclusions

In [30]:
all_exclusions <- rbind(cohort_1,cohort_2)

In [31]:
exclusions_alltime <- all_exclusions |> 
    group_by(person_id) %>%
      summarise(
        Suspensions = sum(TotalFixedExclusions, na.rm = TRUE),
        SuspensionSessions = sum(TotalFixedSessions, na.rm = TRUE),
        Exclusions = sum(PermanentExclusionCount, na.rm = TRUE),
        .groups = "drop"
      )

In [32]:
exclusions_alltime

person_id,Suspensions,SuspensionSessions,Exclusions
<chr>,<dbl>,<dbl>,<dbl>
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,1,2,0
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,4,21,0
002EF3F17C7981C3D6FD842C559E9AA0626E06B3940B670DDA2B71F7070ADF7D,3,14,0
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,8,36,0
006E9437536191621258B883B68946C2B0D85A0161B4CC2FE397CDE1FC1034FA,1,2,0
00984270C3D5F57B15FBB33EE849BA97F72C4F732D031E981D2C43395F4E5098,1,20,0
00A0BB9E50F2E2C8181D0DACFEFF0F3136172F3133377432BB753099B678A32D,1,2,0
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,26,143,0
00DE1C30EB1A1AA2CE3FB40A6E8F2CD3C88180EE30C0AE6DEB98D3F0D1BD812C,1,1,0


## Save csv

In [33]:
# save as csv
write.csv(exclusions_alltime, "data/exclusions.csv", row.names = FALSE)